In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"

In [ ]:
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

# A Simple Calculator

This file shows the implementation of a *symbolic calculator* using `Larkita`.
The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{stmnt}   & \rightarrow & \;\texttt{IDENTIFIER}\; \texttt{':='} \; \;\texttt{expr}\; \texttt{';'} \\
                   & \mid        & \;\texttt{expr}\; \texttt{';'} \\[0.2cm]
  \texttt{expr}    & \rightarrow & \;\texttt{expr}\; \texttt{'+'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{expr}\; \texttt{'-'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{product}                                  \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{product}\; \texttt{'*'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{product}\; \texttt{'/'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{factor}                                   \\[0.2cm]
  \texttt{factor}  & \rightarrow &   \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \;\texttt{NUMBER}                                   \\
                   & \mid        & \;\texttt{IDENTIFIER}
\end{array}
$$

## Setup and Imports
We load the F# port of `Larkita`, which is stored in the directory `../Support/Larkita`.  It generates a *canonical LR(1) parser* that returns the abstract syntax tree of its input as a *nested tuple*.  

In [ ]:
#load "../Support/Larkita/Larkita.fsx"
open Larkita
open LarkitaAst

In Python, `Larkita` returns the abstract syntax tree as a *nested tuple* whose first element is the *head*.
F# has no tuples of varying length, therefore the F# port of `Larkita` (`../Support/Larkita/Larkita.fsx`)
represents these tuples by the discriminated union `Ast` of the module `LarkitaAst`:
```fsharp
type Ast =
    | Node of string list * Ast list   // (head, child_1, ..., child_n)
    | Tok  of string * string          // a named token like ('NUMBER', '42')
    | Lit  of string                   // a literal like '-' that is a child, see 01-Calculator-EBNF.ipynb
```
The head is a list of strings:  a head consisting of a single literal like `'+'` becomes `["+"]`, and a head that
is a tuple of several literals like `(':=', ';')` becomes `[":="; ";"]`.  The function `toString` writes an `Ast` in
Python's notation.

| Python                                     | F#                                                      |
|--------------------------------------------|---------------------------------------------------------|
| `('+', l, r)`                              | `Node(["+"], [l; r])`                                   |
| `((':=', ';'), ('IDENTIFIER', x), e)`      | `Node([":="; ";"], [Tok("IDENTIFIER", x); e])`          |
| `('NUMBER', '42')`                         | `Tok("NUMBER", "42")`                                   |
| `('program',)`                             | `Node(["program"], [])`                                 |
| `'-'` (as a child)                         | `Lit "-"`                                               |

## Specification of the Grammar

`Larkita` reads grammars in the *extended Backus-Naur Form* (EBNF) of `Lark`:  both the lexical tokens and the
grammar rules are defined in a single grammar.  It is stored in the file `calc.lark`, which the cell below displays.
Let us break down the components of our grammar definition:

* `stmnt`, `expr`, `product`, `factor`: These variables define the hierarchical structure of our mathematical
   expressions, natively enforcing operator precedence (e.g., multiplication in `product` happens before addition
   in `expr`).  The first rule of the grammar defines the *start symbol*, hence a `start` rule is not needed.
* **The abstract syntax tree** is determined by the grammar itself:  the string literals of an alternative form
   the *head* of the tuple that is returned, the remaining symbols become its *children*.
   * `expr "+" product` yields `('+', ast(expr), ast(product))`.  Here `ast(expr)` is the *abstract syntax tree* produced when parsing `expr`.
   * `IDENTIFIER ":=" expr ";"` yields `((':=', ';'), ('IDENTIFIER', name), ast(expr))`:  If there is more than
     one literal, the head is the tuple of all literals.
   * `expr ";"` yields `(';', ast(expr))`.
   * A named token like `NUMBER` yields a pair like `('NUMBER', '42')`.
   * An alternative that consists of a single symbol, like `expr: product`, passes the abstract syntax tree of this
     symbol through.  This is what the `?` prefix achieves in `Lark`.
   In F#, `('+', ast(expr), ast(product))` is the value `Node(["+"], [ast(expr); ast(product)])`.
* `%ignore "(", ")"` at the end of an alternative removes these literals from the head.  Then the alternative
  `"(" expr ")"` consists of a single symbol and returns the abstract syntax tree of `expr`.
* **Lexical Imports** (`%import common...`): Instead of manually writing regular expressions for common tokens,
   we import standard terminal definitions.  Several imports are separated by commas.
   * `CNAME` parses standard variable names (letters, numbers, underscores) and is aliased to `IDENTIFIER`.
   * `NUMBER` handles both integers and floating-point numbers.
   * `WS` handles whitespace.
* `%ignore WS`: This directs the lexer to completely discard spaces and tabs so they do not clutter our parsing logic.

The next cell displays the grammar that is stored in `calc.lark`.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "calc.lark")

We generate the parser.  `Larkita` always generates a canonical LR(1) parser.  Furthermore, it writes a description
of all LR(1) states into the file specified by `outFile`.  We will look at this file at the end of this notebook.

In [ ]:
let parser = Larkita.Open("calc.lark", outFile = "calc.out")

# The Abstract Syntax Tree

The parser returns the abstract syntax tree as a nested tuple, so there is no need to convert a parse tree.
For example, the string `x := y + z;` is parsed into the tuple
`((':=', ';'), ('IDENTIFIER', 'x'), ('+', ('IDENTIFIER', 'y'), ('IDENTIFIER', 'z')))`.
In F#, this is the value
```fsharp
Node([":="; ";"], [Tok("IDENTIFIER", "x"); Node(["+"], [Tok("IDENTIFIER", "y"); Tok("IDENTIFIER", "z")])])
```

In [ ]:
let tree = parse parser "x := y + z;"
printfn "%s" (toString tree)
checkNotebook (tree = Node([":="; ";"], [Tok("IDENTIFIER", "x"); Node(["+"], [Tok("IDENTIFIER", "y"); Tok("IDENTIFIER", "z")])]))

In [ ]:
printfn "%s" (toString (parse parser "(1 + 2) * x;"))

## Evaluating the Abstract Syntax Tree

Now we can write our own recursive evaluation function that works directly on the abstract syntax trees returned
by the parser.  The immutable `names2Values` map associates variable names with values.  Each assignment returns an
updated map; a fresh calculation session starts with an empty map.

In [ ]:
let names2Values : Map<string, float> = Map.empty

The Python function `evaluate_adt` returns a number for an expression and nothing for a statement.  In F#,
`evaluate` computes the value of an expression, while `execute` executes a statement and returns the updated map
together with the value of an expression statement.  The patterns are the same as in Python:  for example, the Python
pattern `((':=', ';'), ('IDENTIFIER', var_name), expr)` becomes `Node([":="; ";"], [Tok("IDENTIFIER", name); expr])`.
As in Python, the type `Ast` does not restrict the shape of a tree, so both functions need a final case that reports
an unknown structure.

In [ ]:
let rec evaluate names (adt: Ast) : float =
    match adt with
    | Tok("NUMBER", value) -> System.Double.Parse(value, System.Globalization.CultureInfo.InvariantCulture)
    | Tok("IDENTIFIER", name) ->
        match Map.tryFind name names with
        | Some value -> value
        | None -> failwithf "Undefined variable: %s" name
    | Node([ ("+" | "-" | "*" | "/") as op ], [ left; right ]) ->
        let l, r = evaluate names left, evaluate names right
        match op with
        | "+" -> l + r
        | "-" -> l - r
        | "*" -> l * r
        | _ -> divide l r
    | _ -> failwithf "Unknown ADT structure: %s" (toString adt)

let execute names (adt: Ast) =
    match adt with
    | Node([ ":="; ";" ], [ Tok("IDENTIFIER", name); expr ]) -> Map.add name (evaluate names expr) names, None
    | Node([ ";" ], [ expr ]) -> names, Some(evaluate names expr)
    | _ -> failwithf "Unknown ADT structure: %s" (toString adt)

let calculate names source = execute names (parse parser source)

## Execution

The REPL (Read-Eval-Print Loop) takes the input string, parses it into an abstract syntax tree, and then evaluates it.

In [ ]:
// Kept separate from the demonstrations so running all cells never blocks for input.
let main () =
    let rec loop names =
        printf "calc> "
        let source = System.Console.ReadLine()
        if not (System.String.IsNullOrEmpty source) then
            try
                let updated, value = calculate names source
                value |> Option.iter (printfn "%g")
                loop updated
            with
            | UnexpectedInput e -> printfn "Error: %s" e.Message; loop names
            | error -> printfn "Error: %s" error.Message; loop names
    loop Map.empty

Running all cells never blocks for input; the following cells calculate with fixed inputs instead.

In [ ]:
let assigned, _ = calculate names2Values "x := 6;"
let names, result = calculate assigned "x * 7;"
checkNotebook (result = Some 42.0)
printfn "%A" result
try calculate names "(1 + 2;" |> ignore
with UnexpectedInput e -> printfn "Error: %s" e.Message

Check the populated memory map:

In [ ]:
printfn "%A" names

We can even get the parser states:  `Larkita` has written them into the file `calc.out`.  This file lists

1. the rules of the grammar, each followed by the abstract syntax tree that a reduction with this rule produces,
2. the conflicts (there are none), and
3. all states of the canonical LR(1) automaton.  Every state lists its marked rules together with their
   *lookahead* tokens, followed by the actions and the gotos of the state.  The marked rules that are added
   by the closure are marked with a `+`.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "calc.out")